# Imports

In [22]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

# Read the datasets

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
recipes_df = pd.read_parquet("/content/drive/MyDrive/Health Oriented Food Recommendation System/data/recipes.parquet")
reviews_df = pd.read_parquet("/content/drive/MyDrive/Health Oriented Food Recommendation System/data/reviews.parquet")

# Display

In [25]:
recipes_df.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38.0,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09 21:46:00+00:00,Make and share this Low-Fat Berry Blue Frozen ...,[https://img.sndimg.com/food/image/upload/w_55...,...,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,None,"[Toss 2 cups berries with sugar., Let stand fo..."
1,39.0,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29 13:12:00+00:00,Make and share this Biryani recipe from Food.com.,[https://img.sndimg.com/food/image/upload/w_55...,...,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,None,[Soak saffron in warm milk for 5 minutes and p...
2,40.0,Best Lemonade,1566,Stephen Little,PT5M,PT30M,PT35M,1999-09-05 19:52:00+00:00,This is from one of my first Good House Keepi...,[https://img.sndimg.com/food/image/upload/w_55...,...,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0,None,"[Into a 1 quart Jar with tight fitting lid, pu..."
3,41.0,Carina's Tofu-Vegetable Kebabs,1586,Cyclopz,PT20M,PT24H,PT24H20M,1999-09-03 14:54:00+00:00,This dish is best prepared a day in advance to...,[https://img.sndimg.com/food/image/upload/w_55...,...,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0,4 kebabs,"[Drain the tofu, carefully squeezing out exces..."
4,42.0,Cabbage Soup,1538,Duckie067,PT30M,PT20M,PT50M,1999-09-19 06:19:00+00:00,Make and share this Cabbage Soup recipe from F...,[https://img.sndimg.com/food/image/upload/w_55...,...,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0,None,"[Mix everything together and bring to a boil.,..."


In [26]:
reviews_df.head()

,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25 21:44:00+00:00,2000-01-25 21:44:00+00:00
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17 16:49:59+00:00,2001-10-17 16:49:59+00:00
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25 09:00:00+00:00,2000-02-25 09:00:00+00:00
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13 21:15:00+00:00,2000-03-13 21:15:00+00:00
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28 12:51:00+00:00,2000-03-28 12:51:00+00:00


# Pre-processing

In [27]:
nutritional_cols = ["RecipeId", "Name", "Calories", "FatContent", "SaturatedFatContent", "CholesterolContent", "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent", "ProteinContent", "RecipeServings"]
nutritional_df = recipes_df[nutritional_cols]

In [28]:
nutritional_df.head()

,RecipeId,Name,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings
0,38.0,Low-Fat Berry Blue Frozen Dessert,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0
1,39.0,Biryani,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0
2,40.0,Best Lemonade,311.1,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0
3,41.0,Carina's Tofu-Vegetable Kebabs,536.1,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0
4,42.0,Cabbage Soup,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0


In [29]:
nutritional_df = nutritional_df.dropna(axis=0).reset_index(drop=True)
nutritional_df['RecipeId'] = np.arange(1, len(nutritional_df) + 1)
nutritional_df.head()

,RecipeId,Name,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings
0,1,Low-Fat Berry Blue Frozen Dessert,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0
1,2,Biryani,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0
2,3,Best Lemonade,311.1,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3,4.0
3,4,Carina's Tofu-Vegetable Kebabs,536.1,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3,2.0
4,5,Cabbage Soup,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3,4.0


Divide all columns in nutritional_df by the by RecipeServings

In [30]:
cols_to_divide = ["Calories", "FatContent", "SaturatedFatContent", "CholesterolContent", "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent", "ProteinContent"]
nutritional_df.drop("RecipeServings", axis=1)

,RecipeId,Name,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent
0,1,Low-Fat Berry Blue Frozen Dessert,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2
1,2,Biryani,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4
2,3,Best Lemonade,311.1,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3
3,4,Carina's Tofu-Vegetable Kebabs,536.1,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3
4,5,Cabbage Soup,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3
...,...,...,...,...,...,...,...,...,...,...,...
339601,339602,Spanish Coffee with Tia Maria,84.3,2.1,1.2,6.8,15.7,16.6,0.4,15.4,0.6
339602,339603,Slow-Cooker Classic Coffee Cake,358.9,19.8,10.5,103.1,323.4,41.5,0.8,24.8,4.8
339603,339604,Meg's Fresh Ginger Gingerbread,316.6,12.5,7.6,54.4,278.2,48.5,0.8,22.8,3.9
339604,339605,Roast Prime Rib au Poivre with Mixed Peppercorns,2063.4,172.4,71.4,433.8,766.3,3.2,0.7,0.1,117.0


# Train a kNN model
This identifies the 50 most similar recipes to a given recipe.

In [31]:
class RecipeRecommender:
    def __init__(self, nutritional_df, cols_to_divide):
        self.original_nutritional_df = nutritional_df
        self.nutritional_df = nutritional_df.copy()
        self.cols_to_divide = cols_to_divide

        # normalize the attributes
        self.scaler = MinMaxScaler()
        self.nutritional_df[self.cols_to_divide] = self.scaler.fit_transform(self.nutritional_df[self.cols_to_divide])

        self.knn = NearestNeighbors(metric='euclidean')
        self.knn.fit(self.nutritional_df[self.cols_to_divide])

    def find_closest_recipes(self, recipe_id, k=50):
        input_recipe = self.nutritional_df.loc[self.nutritional_df["RecipeId"] == recipe_id, self.cols_to_divide]
        distances, indices = self.knn.kneighbors(input_recipe, n_neighbors=k+1)  # +1 to exclude the recipe itself

        closest_indices = indices[0][1:]  # Exclude the first element (recipe itself)
        return self.original_nutritional_df.iloc[closest_indices]

    def get_trained_model(self):
        return self.knn

In [32]:
# Initialize the RecipeRecommender with the prepared nutritional_df
recommender = RecipeRecommender(nutritional_df, cols_to_divide)

# Healthiness score

In [33]:
simplified_cols = ["Calories", "FatContent", "SaturatedFatContent", "CholesterolContent", "SodiumContent", "CarbohydrateContent", "FiberContent", "SugarContent", "ProteinContent"]
simplified_df = recipes_df[simplified_cols]
simplified_df

,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent
0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2
1,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4
2,311.1,0.2,0.0,0.0,1.8,81.5,0.4,77.2,0.3
3,536.1,24.0,3.8,0.0,1558.6,64.2,17.3,32.1,29.3
4,103.6,0.4,0.1,0.0,959.3,25.1,4.8,17.7,4.3
...,...,...,...,...,...,...,...,...,...
522512,316.6,12.5,7.6,54.4,278.2,48.5,0.8,22.8,3.9
522513,2063.4,172.4,71.4,433.8,766.3,3.2,0.7,0.1,117.0
522514,1271.3,117.2,72.6,470.9,192.5,33.9,0.0,17.3,12.8
522515,16.1,0.6,0.1,2.9,100.5,0.3,0.0,0.1,2.4


In [34]:
# Calculate and print the average of each column
for column in simplified_df:
  average = simplified_df[column].mean()
  print(f"Average of '{column}': {average}")

Average of 'Calories': 484.4385799887851
Average of 'FatContent': 24.614921811156375
Average of 'SaturatedFatContent': 9.559457204263213
Average of 'CholesterolContent': 86.48700310229141
Average of 'SodiumContent': 767.2638783044381
Average of 'CarbohydrateContent': 49.08909241230428
Average of 'FiberContent': 3.8432420380580914
Average of 'SugarContent': 21.878253913269813
Average of 'ProteinContent': 17.469510274306863


In [35]:
def compute_healthiness_score(recipe):
    # Define the weights for each attribute
    weights = {
        'Calories': -0.002, # 1 / Average of 'Calories'
        'FatContent': -0.04,
        'SaturatedFatContent': -0.1,
        'CholesterolContent': -0.01,
        'SodiumContent': -0.0015,
        'CarbohydrateContent': -0.02,
        'FiberContent': 0.9,  # 3.5 coefficient for positive weights
        'SugarContent': -0.045,
        'ProteinContent': 0.2
    }

    # Compute the score using the formula
    score = sum(recipe[attribute] * weights[attribute] for attribute in weights)

    return score

# Testing the model

In [36]:
# Find the closest recipes for a given input recipe
input_recipe_id = 123456 # enter the input recipe number here (choose a number from 1 to 339606)

def print_recipe_name(recipe_id):
    recipe_name = nutritional_df.loc[nutritional_df['RecipeId'] == recipe_id, 'Name'].values[0]
    print(f"Selected Recipe: {recipe_name}")

print_recipe_name(input_recipe_id)

result = recommender.find_closest_recipes(input_recipe_id)
print("Closest recipes for RecipeId", input_recipe_id, "from the original nutritional_df:")
result

Selected Recipe: Spinach and Tomatoes
Closest recipes for RecipeId 123456 from the original nutritional_df:


,RecipeId,Name,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings
267692,267693,Broccoli With Truffle Oil,110.6,1.2,0.1,0.0,112.7,21.5,8.0,5.2,8.8,4.0
5264,5265,Easy Pasta Con Broccoli,105.3,1.1,0.1,0.0,113.6,20.6,8.0,5.3,8.7,4.0
327117,327118,Tender and Spicy Green Soup,102.2,1.4,0.2,0.0,201.1,18.9,8.2,6.3,7.8,2.0
284729,284730,Charred Broccoli,121.3,2.2,0.3,0.0,178.7,22.5,8.4,5.6,8.9,2.0
180331,180332,Spicy Baked Beans,140.5,0.7,0.1,0.0,107.2,26.3,8.0,2.8,8.7,8.0
32021,32022,Black Beans with Peppers and Mushrooms,148.6,2.1,0.3,0.0,144.6,25.6,8.3,3.7,9.2,4.0
188917,188918,Sicilian Gnocchi With Broccoli,109.1,1.1,0.1,0.0,136.5,21.9,7.8,6.4,8.3,4.0
95755,95756,Veggie and Black Bean Wrap,127.5,0.7,0.1,0.0,27.0,23.2,8.1,1.9,9.0,4.0
84259,84260,Pimiento Puree,115.8,1.2,0.3,1.2,91.5,22.2,7.9,8.4,9.4,2.0
111174,111175,Wrapped Vegetables,100.8,1.0,0.3,0.0,46.4,18.2,8.2,4.3,9.6,4.0


In [37]:
recipe_df = nutritional_df.loc[nutritional_df["RecipeId"] == input_recipe_id]
recipe_df["healthiness_score"] = recipe_df.apply(compute_healthiness_score, axis=1)
input_recipe_score = recipe_df[["RecipeId", "Name", "healthiness_score"]]
input_recipe_score

/tmp/ipython-input-2680421106.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recipe_df["healthiness_score"] = recipe_df.apply(compute_healthiness_score, axis=1)


,RecipeId,Name,healthiness_score
123455,123456,Spinach and Tomatoes,7.99645


In [38]:
# Compute healthiness scores for each recipe
result["healthiness_score"] = result.apply(compute_healthiness_score, axis=1)

# Sort recipes based on healthiness scores and get the top 8
top_8_recipes = result.nlargest(8, "healthiness_score")[["RecipeId", "Name", "healthiness_score"]]

top_8_recipes

/tmp/ipython-input-2171897526.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result["healthiness_score"] = result.apply(compute_healthiness_score, axis=1)


,RecipeId,Name,healthiness_score
13249,13250,Light Refried Black Beans,8.57850
11256,11257,Lemon Asparagus II,8.57190
111174,111175,Wrapped Vegetables,8.40130
5529,5530,Awesome Pinto Beans,8.38020
46817,46818,"Spinach, Lemon and Lentil Soup",8.30365
283082,283083,Bean Thickened Soup,8.28935
296629,296630,Bean Thickened Soup,8.28935
262304,262305,Simple No Fuss Raspberry Carrot Snack (Vegan),8.27135
